In [1]:
import pyspark


Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
0,application_1784539943449_0001,pyspark3,idle,Link,Link,✔


SparkSession available as 'spark'.


In [2]:
df = spark.read.parquet('s3://airline-dataset-2020-2025/Silver/')

In [3]:
df.show()

+-------+-----+----------+---------+-------------------+-------------------------+---------------------------------------+------------------------+---------------------------+-------------------------------+---------------------------------------+----------------------------------------------+-------------------------------------------------+--------------------------------------------------+-----------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+------+--------------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+----+-----------------+---------+-------------+-------------+-------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+----------+-------+--------+---------------+--------+------------------+----------+---------+----------------+--------

In [57]:
print(df.columns)

['Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Marketing_Airline_Network', 'Operated_or_Branded_Code_Share_Partners', 'DOT_ID_Marketing_Airline', 'IATA_Code_Marketing_Airline', 'Flight_Number_Marketing_Airline', 'Originally_Scheduled_Code_Share_Airline', 'DOT_ID_Originally_Scheduled_Code_Share_Airline', 'IATA_Code_Originally_Scheduled_Code_Share_Airline', 'Flight_Num_Originally_Scheduled_Code_Share_Airline', 'Operating_Airline', 'DOT_ID_Operating_Airline', 'IATA_Code_Operating_Airline', 'Tail_Number', 'Flight_Number_Operating_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'Tax

# Checking the null values 

In [59]:
from pyspark.sql.functions import col, sum, when

row = df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).first()

for column in df.columns:
    if row[column] > 0:
        print(f"{column}: {row[column]}")

Flight_Number_Marketing_Airline: 1
Originally_Scheduled_Code_Share_Airline: 40906561
DOT_ID_Originally_Scheduled_Code_Share_Airline: 40906561
IATA_Code_Originally_Scheduled_Code_Share_Airline: 40906561
Flight_Num_Originally_Scheduled_Code_Share_Airline: 40906561
Tail_Number: 296073
Flight_Number_Operating_Airline: 1
DepTime: 895495
DepDelay: 896518
DepDelayMinutes: 896518
DepDel15: 896518
DepartureDelayGroups: 896518
TaxiOut: 912686
WheelsOff: 912686
WheelsOn: 926416
TaxiIn: 926416
ArrTime: 926392
ArrDelay: 1014879
ArrDelayMinutes: 1014879
ArrDel15: 1014879
ArrivalDelayGroups: 1014879
CancellationCode: 39993169
CRSElapsedTime: 14
ActualElapsedTime: 1014879
AirTime: 1014879
CarrierDelay: 33265986
WeatherDelay: 33265986
NASDelay: 33265986
SecurityDelay: 33265986
LateAircraftDelay: 33265986
FirstDepTime: 40639472
TotalAddGTime: 40639523
LongestAddGTime: 40639526
DivAirportLandings: 96
DivReachedDest: 40812463
DivActualElapsedTime: 40821825
DivArrDelay: 40821767
DivDistance: 40812463
Div1A

# Selecting which columns are eligible (leaning to our problem statement)

In [60]:
from pyspark.sql.functions import col, sum, when

# Select required columns
df1 = df.select(
    'FlightDate',
    'Year',
    'Month',
    'Quarter',
    'DayOfMonth',
    'DayOfWeek',
    'Marketing_Airline_Network',
    'Flight_Number_Marketing_Airline',
    'Origin',
    'OriginState',
    'Dest',
    'DestState',
    'CRSDepTime',
    'CRSArrTime',
    'ArrDelay',
    'ArrDel15',
    'DepDelay',
    'DepDel15',
    'CarrierDelay',
    'WeatherDelay',
    'NASDelay',
    'SecurityDelay',
    'LateAircraftDelay',
    'AirTime',
    'Cancelled',
    'Diverted',
    'Distance',
    'TaxiOut',
    'TaxiIn'
)

# Count NULL values
null_counts = df1.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df1.columns
])

null_counts.show(truncate=False)


+----------+----+-----+-------+----------+---------+-------------------------+-------------------------------+------+-----------+----+---------+----------+----------+--------+--------+--------+--------+------------+------------+--------+-------------+-----------------+-------+---------+--------+--------+-------+------+
|FlightDate|Year|Month|Quarter|DayOfMonth|DayOfWeek|Marketing_Airline_Network|Flight_Number_Marketing_Airline|Origin|OriginState|Dest|DestState|CRSDepTime|CRSArrTime|ArrDelay|ArrDel15|DepDelay|DepDel15|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|AirTime|Cancelled|Diverted|Distance|TaxiOut|TaxiIn|
+----------+----+-----+-------+----------+---------+-------------------------+-------------------------------+------+-----------+----+---------+----------+----------+--------+--------+--------+--------+------------+------------+--------+-------------+-----------------+-------+---------+--------+--------+-------+------+
|0         |0   |0    |0      |0     

## Summary of null values
    1.'FlightDate' - no null
    2.'Year' - no null
    3.'Month' -no null
    4.'Quarter' - no null 
    5.'DayOfMonth' -no null
    6.'DayOfWeek'-no null
    7.'Marketing_Airline_Network'-no null
    8.'Flight_Number_Marketing_Airline' - 1
    9.'Origin'- no null
    10.'OriginState' -no null
    11.'Dest'-no null
    12.'DestState'-no null
    13.'CRSDepTime'-no null
    14. 'CRSArrTime' - no null
    13.'ArrDelay' - 1014897
    14.'ArrDel15' - 1015897
    15.'DepDelay' - 896518
    16.'DepDel15' - 896518
    17.'CarrierDelay'- 33265986
    18.'WeatherDelay'- 33265986
    19.'NASDelay'- 33265986
    20.'SecurityDelay'- 33265986
    21.'LateAircraftDelay'- 33265986
    22.'AirTime' - 1014879
    23.'Cancelled'- 0
    24.'Diverted'- 0
    25.'Distance' - 0
    26.'TaxiOut' - 912686
    27.'TaxiIn' - 912686

In [61]:
df1.printSchema()

root
 |-- FlightDate: timestamp (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- DayOfMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- Flight_Number_Marketing_Airline: integer (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestState: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- ArrDel15: double (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- DepDel15: double (nullable = true)
 |-- CarrierDelay: double (nullable = true)
 |-- WeatherDelay: double (nullable = true)
 |-- NASDelay: double (nullable = true)
 |-- SecurityDelay: double (nullable = true)
 |-- LateAircraftDelay: double (nullable = true)
 |-

# unique values of the categorical columns

In [9]:
from pyspark.sql.functions import countDistinct

categorical_cols = [
    "Marketing_Airline_Network",
    "Origin",
    "OriginState",
    "Dest",
    "DestState",
    "Cancelled",
    "Diverted",
    "DepDel15",
    "ArrDel15"
]

unique_counts = df.select([
    countDistinct(c).alias(c)
    for c in categorical_cols
])

unique_counts.show(truncate=False)

+-------------------------+------+-----------+----+---------+---------+--------+--------+--------+
|Marketing_Airline_Network|Origin|OriginState|Dest|DestState|Cancelled|Diverted|DepDel15|ArrDel15|
+-------------------------+------+-----------+----+---------+---------+--------+--------+--------+
|10                       |390   |53         |391 |53       |2        |2       |2       |2       |
+-------------------------+------+-----------+----+---------+---------+--------+--------+--------+

In [10]:
origin_not_in_dest = (
    df.select("Origin")
      .distinct()
      .subtract(df.select("Dest").distinct())
)

origin_not_in_dest.show(truncate=False)

+------+
|Origin|
+------+
+------+

## destination Airport IFP is distinct 

In [11]:
dest_not_in_origin = (
    df.select("Dest")
      .distinct()
      .subtract(df.select("Origin").distinct())
)

dest_not_in_origin.show(truncate=False)

+----+
|Dest|
+----+
|IFP |
+----+

In [12]:
from pyspark.sql.functions import countDistinct

df.select(countDistinct("FlightDate").alias("Distinct_Flight_Dates")).show()

+---------------------+
|Distinct_Flight_Dates|
+---------------------+
|                 2192|
+---------------------+

In [13]:
from pyspark.sql.functions import countDistinct

# Distinct FlightDate count
flightdate_count = df.select("FlightDate").distinct().count()

# Distinct Year-Month-DayOfMonth combinations
ymd_count = df.select(
    "Year",
    "Month",
    "DayOfMonth"
).distinct().count()

print(f"Distinct FlightDate count      : {flightdate_count}")
print(f"Distinct Year-Month-Day count  : {ymd_count}")

if flightdate_count == ymd_count:
    print("✅ Counts are the SAME.")
else:
    print("❌ Counts are DIFFERENT.")

Distinct FlightDate count      : 2192
Distinct Year-Month-Day count  : 2192
✅ Counts are the SAME.

## we can either use flight date or year, month, day of week for analysis


In [14]:
route_count = df.select("Origin", "Dest").distinct().count()

print("Unique Origin-Dest combinations:", route_count)

Unique Origin-Dest combinations: 8734

## unique routes 

In [3]:
#distributed dataframe 
df1 = df.select('Year','Flight_Number_Operating_Airline','DayofWeek','FlightDate','CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay')

In [4]:
# there are many null values. #we are interested in the delay, hence consider only the rows having either of the delay, 
# drop the null values for that 

delayed_df = df1.dropna()

In [5]:
delayed_df.show()

+----+-------------------------------+---------+-------------------+------------+------------+--------+-------------+-----------------+
|Year|Flight_Number_Operating_Airline|DayofWeek|         FlightDate|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|
+----+-------------------------------+---------+-------------------+------------+------------+--------+-------------+-----------------+
|2021|                            683|        4|2021-07-15 00:00:00|         0.0|         0.0|    18.0|          0.0|              0.0|
|2021|                            688|        4|2021-07-15 00:00:00|         0.0|         0.0|     0.0|          0.0|             52.0|
|2021|                            729|        4|2021-07-15 00:00:00|         0.0|        12.0|     5.0|          0.0|              0.0|
|2021|                            756|        4|2021-07-15 00:00:00|         0.0|         0.0|    15.0|          0.0|              0.0|
|2021|                            781|        4|

# total count of the delay cause 

In [6]:
delayed_df.count()

7644267

# yearwise pattern of total delay cause

In [7]:
# year wise delay
delayed_df.groupBy("Year").count().orderBy("Year").show()

+----+-------+
|Year|  count|
+----+-------+
|2020| 468954|
|2021|1068058|
|2022|1426305|
|2023|1464538|
|2024|1531078|
|2025|1685334|
+----+-------+

# total individual delay count

In [8]:
columns = [
    'CarrierDelay',
    'WeatherDelay',
    'NASDelay',
    'SecurityDelay',
    'LateAircraftDelay'
]

for c in columns:
    count = delayed_df.filter(delayed_df[c] > 0.0).count()
    print(f"Number of flights with {c} > 0: {count}")

Number of flights with CarrierDelay > 0: 4256432
Number of flights with WeatherDelay > 0: 462135
Number of flights with NASDelay > 0: 3703330
Number of flights with SecurityDelay > 0: 38359
Number of flights with LateAircraftDelay > 0: 3738361

In [9]:
from pyspark.sql.functions import col

delay_summary = delayed_df.filter(
    (col("CarrierDelay") > 0) |
    (col("WeatherDelay") > 0) |
    (col("SecurityDelay") > 0) |
    (col("NASDelay") > 0) |
    (col("LateAircraftDelay") > 0)
).select(
    "CarrierDelay",
    "WeatherDelay",
    "SecurityDelay",
    "NASDelay",
    "LateAircraftDelay"
).summary()

delay_summary.show(truncate=False)

+-------+-----------------+-----------------+-------------------+------------------+------------------+
|summary|CarrierDelay     |WeatherDelay     |SecurityDelay      |NASDelay          |LateAircraftDelay |
+-------+-----------------+-----------------+-------------------+------------------+------------------+
|count  |7644250          |7644250          |7644250            |7644250           |7644250           |
|mean   |25.26401321254538|4.279225038427576|0.13603466657945515|13.147283382934885|27.125064852667037|
|stddev |75.45415899735512|34.04625979608069|3.522512840205176  |31.962146329431757|61.031560792948866|
|min    |0.0              |0.0              |0.0                |0.0               |0.0               |
|25%    |0.0              |0.0              |0.0                |0.0               |0.0               |
|50%    |4.0              |0.0              |0.0                |0.0               |0.0               |
|75%    |23.0             |0.0              |0.0                

##### The delay cause is rightly skewed

# summary of individual delay cause and  columns 

In [17]:
c_delay_df = delayed_df.filter(delayed_df.CarrierDelay > 0.0).select('CarrierDelay')
c_delay_df.summary().show()

+-------+-----------------+
|summary|     CarrierDelay|
+-------+-----------------+
|  count|          4256432|
|   mean|  45.372375971236|
| stddev|96.50105571333697|
|    min|              1.0|
|    25%|              9.0|
|    50%|             20.0|
|    75%|             44.0|
|    max|           7232.0|
+-------+-----------------+

In [18]:
w_delay_df = delayed_df.filter(delayed_df.WeatherDelay > 0.0).select('WeatherDelay')
w_delay_df.summary().show()

+-------+------------------+
|summary|      WeatherDelay|
+-------+------------------+
|  count|            462135|
|   mean| 70.78335551299945|
| stddev|120.27591269990216|
|    min|               1.0|
|    25%|              15.0|
|    50%|              34.0|
|    75%|              78.0|
|    max|            2419.0|
+-------+------------------+

In [19]:
n_delay_df = delayed_df.filter(delayed_df.NASDelay > 0.0).select('NASDelay')
n_delay_df.summary().show()

+-------+-----------------+
|summary|         NASDelay|
+-------+-----------------+
|  count|          3703330|
|   mean|27.13804089832664|
| stddev|41.58144075346053|
|    min|              1.0|
|    25%|              8.0|
|    50%|             17.0|
|    75%|             30.0|
|    max|           2700.0|
+-------+-----------------+

In [20]:
s_delay_df = delayed_df.filter(delayed_df.SecurityDelay > 0.0).select('SecurityDelay')
s_delay_df.summary().show()

+-------+------------------+
|summary|     SecurityDelay|
+-------+------------------+
|  count|             38359|
|   mean|27.109231210406946|
| stddev|41.731644101729955|
|    min|               1.0|
|    25%|              10.0|
|    50%|              18.0|
|    75%|              30.0|
|    max|            1460.0|
+-------+------------------+

In [21]:
l_delay_df = delayed_df.filter(delayed_df.LateAircraftDelay > 0.0).select('LateAircraftDelay')
l_delay_df.summary().show()

+-------+------------------+
|summary| LateAircraftDelay|
+-------+------------------+
|  count|           3738361|
|   mean|55.465691248116485|
| stddev| 77.74762764641166|
|    min|               1.0|
|    25%|              16.0|
|    50%|              32.0|
|    75%|              67.0|
|    max|            3581.0|
+-------+------------------+

In [23]:
df2 =df.select('Origin','Dest','CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay')

# top 5 routes having delay cause

In [27]:
from pyspark.sql.functions import when, count

route_reliability = df2.groupBy("Origin", "Dest").agg(
    count("*").alias("TotalFlights"),

    count(when(col("CarrierDelay") > 0, True)).alias("CarrierDelayFlights"),
    count(when(col("WeatherDelay") > 0, True)).alias("WeatherDelayFlights"),
    count(when(col("SecurityDelay") > 0, True)).alias("SecurityDelayFlights"),
    count(when(col("NASDelay") > 0, True)).alias("NASDelayFlights"),
    count(when(col("LateAircraftDelay") > 0, True)).alias("LateAircraftDelayFlights")
)

route_reliability.orderBy(route_reliability['TotalFlights'], ascending = False).show(5, truncate=False)

+------+----+------------+-------------------+-------------------+--------------------+---------------+------------------------+
|Origin|Dest|TotalFlights|CarrierDelayFlights|WeatherDelayFlights|SecurityDelayFlights|NASDelayFlights|LateAircraftDelayFlights|
+------+----+------------+-------------------+-------------------+--------------------+---------------+------------------------+
|LAX   |SFO |65140       |5491               |391                |22                  |7567           |5434                    |
|SFO   |LAX |65091       |5535               |265                |17                  |4036           |6593                    |
|HNL   |OGG |60102       |6324               |282                |61                  |1131           |4398                    |
|OGG   |HNL |60101       |6470               |466                |41                  |2056           |7087                    |
|LAX   |LAS |57531       |5999               |270                |57                  |6770      

In [29]:
df3 = df.select('CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay','ArrDelay','DepDelay','Cancelled','Diverted')

# Flights which are delayed but no reason mentioned  

In [37]:
cause = df3.filter(((df3.ArrDelay >0.0) & (df3.DepDelay >0.0))& df3.CarrierDelay.isNull())
cause.count()

3264755

In [38]:
cause.show()

+------------+------------+--------+-------------+-----------------+--------+--------+---------+--------+
|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|ArrDelay|DepDelay|Cancelled|Diverted|
+------------+------------+--------+-------------+-----------------+--------+--------+---------+--------+
|        null|        null|    null|         null|             null|     6.0|     8.0|      0.0|     0.0|
|        null|        null|    null|         null|             null|     6.0|    15.0|      0.0|     0.0|
|        null|        null|    null|         null|             null|     2.0|    13.0|      0.0|     0.0|
|        null|        null|    null|         null|             null|    10.0|    10.0|      0.0|     0.0|
|        null|        null|    null|         null|             null|    12.0|    15.0|      0.0|     0.0|
|        null|        null|    null|         null|             null|     9.0|    15.0|      0.0|     0.0|
|        null|        null|    null|         n

In [42]:
display = cause.filter(
    (cause.Cancelled == 0.0) & (cause.Diverted == 0.0)
)

display.select(
    'ArrDelay',
    'DepDelay',
    'Cancelled',
    'Diverted'
).show()

+--------+--------+---------+--------+
|ArrDelay|DepDelay|Cancelled|Diverted|
+--------+--------+---------+--------+
|     6.0|     8.0|      0.0|     0.0|
|     6.0|    15.0|      0.0|     0.0|
|     2.0|    13.0|      0.0|     0.0|
|    10.0|    10.0|      0.0|     0.0|
|    12.0|    15.0|      0.0|     0.0|
|     9.0|    15.0|      0.0|     0.0|
|     4.0|    16.0|      0.0|     0.0|
|    13.0|    22.0|      0.0|     0.0|
|     2.0|    19.0|      0.0|     0.0|
|    10.0|    32.0|      0.0|     0.0|
|     7.0|     5.0|      0.0|     0.0|
|    14.0|     4.0|      0.0|     0.0|
|     9.0|     1.0|      0.0|     0.0|
|     4.0|     2.0|      0.0|     0.0|
|     5.0|     1.0|      0.0|     0.0|
|    12.0|     3.0|      0.0|     0.0|
|    13.0|    17.0|      0.0|     0.0|
|    13.0|    17.0|      0.0|     0.0|
|     2.0|     1.0|      0.0|     0.0|
|     1.0|     5.0|      0.0|     0.0|
+--------+--------+---------+--------+
only showing top 20 rows

In [43]:
display.count()

3264755

# Season Wise delay trend

In [34]:
from pyspark.sql.functions import count

month_df = df.select('WeatherDelay','Month')
month_df = month_df.filter(month_df['WeatherDelay'] >0.0)
result = month_df.groupBy('Month').agg(count('WeatherDelay'))
result.orderBy('count(WeatherDelay)',ascending=False).show()

+-----+-------------------+
|Month|count(WeatherDelay)|
+-----+-------------------+
|    7|              65707|
|    6|              56873|
|    8|              52241|
|    1|              43154|
|    5|              42866|
|   12|              42119|
|    2|              34541|
|    3|              29453|
|    4|              29375|
|    9|              26396|
|   11|              20549|
|   10|              18861|
+-----+-------------------+

In [62]:
from pyspark.sql.functions import when, col, count

# Create Season column
season_df = df.withColumn(
    "Season",
    when(col("Month").isin(12, 1, 2), "Winter")
    .when(col("Month").isin(3, 4, 5), "Spring")
    .when(col("Month").isin(6, 7, 8), "Summer")
    .otherwise("Fall")
)

# Season-wise summary for all delay causes
result = season_df.groupBy("Season").agg(
    count(when(col("CarrierDelay") > 0, True)).alias("CarrierDelayCount"),
    count(when(col("WeatherDelay") > 0, True)).alias("WeatherDelayCount"),
    count(when(col("NASDelay") > 0, True)).alias("NASDelayCount"),
    count(when(col("SecurityDelay") > 0, True)).alias("SecurityDelayCount"),
    count(when(col("LateAircraftDelay") > 0, True)).alias("LateAircraftDelayCount")
)

result.show(truncate=False)

+------+-----------------+-----------------+-------------+------------------+----------------------+
|Season|CarrierDelayCount|WeatherDelayCount|NASDelayCount|SecurityDelayCount|LateAircraftDelayCount|
+------+-----------------+-----------------+-------------+------------------+----------------------+
|Spring|991530           |101694           |910633       |9361              |891851                |
|Summer|1366109          |174821           |1117858      |12045             |1263517               |
|Fall  |876410           |65806            |744959       |7657              |727507                |
|Winter|1022383          |119814           |929880       |9296              |855486                |
+------+-----------------+-----------------+-------------+------------------+----------------------+

# daily trend of flight Congestion 

In [35]:
nas_df = df.select('NASDelay','DayOfWeek','')
nas_df = nas_df.filter(nas_df['NASDelay'] >0.0)
result = nas_df.groupBy('DayOfWeek').agg(count('NASDelay'))
result.orderBy('count(NASDelay)',ascending=False).show()

+---------+---------------+
|DayOfWeek|count(NASDelay)|
+---------+---------------+
|        5|         612510|
|        4|         604164|
|        7|         590188|
|        1|         552893|
|        3|         473087|
|        6|         446346|
|        2|         424142|
+---------+---------------+

In [63]:
from pyspark.sql.functions import col, when, count

nas_df = df.select("NASDelay", "DayOfWeek") \
           .filter(col("NASDelay") > 0)

nas_df = nas_df.withColumn(
    "DayType",
    when(col("DayOfWeek").isin(6, 7), "Weekend")
    .otherwise("Weekday")
)

result = nas_df.groupBy("DayType") \
               .agg(count("NASDelay").alias("NASDelayCount")) \
               .orderBy(col("NASDelayCount").desc())

result.show()

+-------+-------------+
|DayType|NASDelayCount|
+-------+-------------+
|Weekday|      2666796|
|Weekend|      1036534|
+-------+-------------+

# finding correlation with every delay

In [65]:
delay_cols = [
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay"
]

print("Correlation Matrix\n")

for i in range(len(delay_cols)):
    for j in range(i + 1, len(delay_cols)):
        corr = df.stat.corr(delay_cols[i], delay_cols[j])
        print(f"{delay_cols[i]:20} vs {delay_cols[j]:20}: {corr:.4f}")

Correlation Matrix

CarrierDelay         vs WeatherDelay        : -0.0040
CarrierDelay         vs NASDelay            : 0.0221
CarrierDelay         vs SecurityDelay       : -0.0007
CarrierDelay         vs LateAircraftDelay   : 0.0604
WeatherDelay         vs NASDelay            : 0.0296
WeatherDelay         vs SecurityDelay       : -0.0004
WeatherDelay         vs LateAircraftDelay   : 0.0254
NASDelay             vs SecurityDelay       : 0.0037
NASDelay             vs LateAircraftDelay   : 0.0453
SecurityDelay        vs LateAircraftDelay   : 0.0047

# check the delay cause matches arrival delay (violating the business or not)

In [5]:
from pyspark.sql.functions import col, abs

violation_count = df.filter(
    (col("ArrDelay") > 0) &
    (
        abs(
            (
                col("CarrierDelay") +
                col("WeatherDelay") +
                col("NASDelay") +
                col("SecurityDelay") +
                col("LateAircraftDelay")
            ) - col("ArrDelay")
        ) > 1
    )
).count()

print(f"Business Rule Violations: {violation_count}")

Business Rule Violations: 17